# Fine-tune FinBERT on financial news headlines

The app labels headlines with [FinBERT](https://huggingface.co/ProsusAI/finbert). On the 2,388
test headlines of [twitter-financial-news-sentiment](https://huggingface.co/datasets/zeroshot/twitter-financial-news-sentiment)
it reaches macro F1 0.668, below a TF-IDF + logistic regression model trained on the same dataset
(0.761; see `evaluation/results.json`). FinBERT was trained on a different set, Financial PhraseBank.
This notebook trains FinBERT further on the dataset's training headlines.

**How the data is used**

- *Train*: 90% of the 9,543 training headlines.
- *Dev*: the other 10% (same label mix). Used to pick the best epoch and the best run.
- *Test*: the 2,388 validation headlines, the same set as in the README table. Never used for any
  choice; scored once per run at the end.

Three runs with different random seeds show how much the result moves by chance. The run with the
best dev score is saved as the model for the app.

**Run on Kaggle:** Settings → Accelerator **GPU T4 x1**, Internet **on**, then *Run all*.

**Outputs** (in `/kaggle/working`):

- `finbert-tweets/`: the saved model and tokenizer;
- `finetune_results.json`: settings, library versions, dev and test scores for each run;
- `test_predictions.csv`: the saved model's label and scores for every test headline.

In [ ]:
import copy
import json
import math
import os
import platform
import random
import time
from collections import Counter
from pathlib import Path

import numpy as np
import torch

SMOKE = os.environ.get("SMOKE") == "1"  # tiny CPU check of the code, not a real run

DATASET = "zeroshot/twitter-financial-news-sentiment"
DATASET_REVISION = "ccbe24de388e287beb92dd393a335c376b350ac3"
BASE = "ProsusAI/finbert"
BASE_REVISION = "0bc2ac10dda6929bcb1f77ccc43c7ec5c424f031"  # the same revision the app uses

SEEDS = [13, 42, 2024]
EPOCHS = 4
LEARNING_RATE = 2e-5
BATCH_SIZE = 32
MAX_TOKENS = 128
WARMUP_SHARE = 0.1
WEIGHT_DECAY = 0.01
DEV_SHARE = 0.1
SPLIT_SEED = 0  # fixes the train/dev split; the same in every run

KAGGLE = Path("/kaggle/working")
OUT = KAGGLE if KAGGLE.exists() else Path(os.environ.get("OUT_DIR", "finetune_outputs"))
OUT.mkdir(parents=True, exist_ok=True)

if SMOKE:
    SEEDS, EPOCHS, BATCH_SIZE = [13], 1, 8
    device = torch.device("cpu")
else:
    if not torch.cuda.is_available():
        raise SystemExit("No GPU. On Kaggle: Settings -> Accelerator -> GPU T4 x1, then run again.")
    device = torch.device("cuda")
USE_FP16 = device.type == "cuda"
print("device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu", "| output folder:", OUT)

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

NAMES = {0: "negative", 1: "positive", 2: "neutral"}  # the dataset's bearish, bullish, neutral
CLASSES = ["negative", "neutral", "positive"]

train_full = load_dataset(DATASET, split="train", revision=DATASET_REVISION)
test_set = load_dataset(DATASET, split="validation", revision=DATASET_REVISION)

all_texts = [" ".join(t.split()) for t in train_full["text"]]  # the app also joins whitespace
all_gold = [NAMES[x] for x in train_full["label"]]
train_idx, dev_idx = train_test_split(range(len(all_texts)), test_size=DEV_SHARE,
                                      random_state=SPLIT_SEED, stratify=all_gold)
train_texts, train_gold = [all_texts[i] for i in train_idx], [all_gold[i] for i in train_idx]
dev_texts, dev_gold = [all_texts[i] for i in dev_idx], [all_gold[i] for i in dev_idx]
test_texts = [" ".join(t.split()) for t in test_set["text"]]
test_gold = [NAMES[x] for x in test_set["label"]]

if SMOKE:
    train_texts, train_gold = train_texts[:64], train_gold[:64]
    dev_texts, dev_gold = dev_texts[:32], dev_gold[:32]
    test_texts, test_gold = test_texts[:48], test_gold[:48]

# Headlines that appear in both the training data and the test set would make the test easier.
seen = {t.lower() for t in all_texts}
overlap = sum(t.lower() in seen for t in test_texts)
print(f"train {len(train_texts)}, dev {len(dev_texts)}, test {len(test_texts)}")
print("label counts  train:", dict(Counter(train_gold)), " dev:", dict(Counter(dev_gold)), " test:", dict(Counter(test_gold)))
print(f"test headlines that also appear in the training data: {overlap}")

In [ ]:
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

tokenizer = AutoTokenizer.from_pretrained(BASE, revision=BASE_REVISION)
base_config = AutoConfig.from_pretrained(BASE, revision=BASE_REVISION)
LABEL2ID = {name.lower(): i for i, name in base_config.id2label.items()}  # FinBERT: positive, negative, neutral
ID2NAME = {i: name for name, i in LABEL2ID.items()}
assert set(LABEL2ID) == set(CLASSES), base_config.id2label


def encode(texts):
    return tokenizer(texts, truncation=True, max_length=MAX_TOKENS)


train_enc, dev_enc, test_enc = encode(train_texts), encode(dev_texts), encode(test_texts)
train_ids = torch.tensor([LABEL2ID[g] for g in train_gold])
lengths = [len(x) for x in tokenizer(train_texts + dev_texts + test_texts)["input_ids"]]
print(f"tokens per headline: median {int(np.median(lengths))}, max {max(lengths)}, "
      f"cut at {MAX_TOKENS}: {sum(n > MAX_TOKENS for n in lengths)} headlines")


def make_batch(enc, rows):
    batch = tokenizer.pad({k: [enc[k][i] for i in rows] for k in enc.keys()}, return_tensors="pt")
    return {k: v.to(device) for k, v in batch.items()}


@torch.no_grad()
def predict(model, enc, batch_size=128):
    # Returns one row of label scores (softmax) per headline, columns in ID2NAME order.
    model.eval()
    n = len(enc["input_ids"])
    out = []
    for start in range(0, n, batch_size):
        batch = make_batch(enc, range(start, min(start + batch_size, n)))
        with torch.autocast(device.type, dtype=torch.float16, enabled=USE_FP16):
            logits = model(**batch).logits
        out.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    return np.concatenate(out)


def labels_of(probs):
    return [ID2NAME[int(i)] for i in probs.argmax(axis=1)]


def scores(gold, pred):
    # The same measures as evaluation/evaluate.py.
    per_class = {}
    for c in CLASSES:
        tp = sum(g == p == c for g, p in zip(gold, pred))
        n_pred, n_gold = pred.count(c), gold.count(c)
        precision = tp / n_pred if n_pred else 0.0
        recall = tp / n_gold if n_gold else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_class[c] = {"precision": round(precision, 4), "recall": round(recall, 4), "f1": round(f1, 4)}
    confusion = {g: {p: sum(1 for a, b in zip(gold, pred) if a == g and b == p) for p in CLASSES} for g in CLASSES}
    return {
        "accuracy": round(sum(g == p for g, p in zip(gold, pred)) / len(gold), 4),
        "macro_f1": round(sum(v["f1"] for v in per_class.values()) / len(CLASSES), 4),
        "per_class": per_class,
        "confusion": confusion,
    }

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def train_one(seed):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(BASE, revision=BASE_REVISION).to(device)
    no_decay = ("bias", "LayerNorm.weight")
    groups = [
        {"params": [p for n, p in model.named_parameters() if not n.endswith(no_decay)], "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in model.named_parameters() if n.endswith(no_decay)], "weight_decay": 0.0},
    ]
    optimizer = torch.optim.AdamW(groups, lr=LEARNING_RATE)
    steps_per_epoch = math.ceil(len(train_texts) / BATCH_SIZE)
    total_steps = EPOCHS * steps_per_epoch
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_SHARE * total_steps), total_steps)
    scaler = torch.amp.GradScaler(device.type, enabled=USE_FP16)
    order = torch.Generator().manual_seed(seed)

    history, best = [], {"dev_macro_f1": -1.0}
    start = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for rows in torch.randperm(len(train_texts), generator=order).split(BATCH_SIZE):
            batch = make_batch(train_enc, rows.tolist())
            with torch.autocast(device.type, dtype=torch.float16, enabled=USE_FP16):
                loss = model(**batch, labels=train_ids[rows].to(device)).loss
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            losses.append(loss.item())
        dev = scores(dev_gold, labels_of(predict(model, dev_enc)))
        history.append({"epoch": epoch, "train_loss": round(float(np.mean(losses)), 4),
                        "dev_accuracy": dev["accuracy"], "dev_macro_f1": dev["macro_f1"]})
        print(f"seed {seed} epoch {epoch}: train loss {history[-1]['train_loss']:.4f}, "
              f"dev accuracy {dev['accuracy']:.4f}, dev macro F1 {dev['macro_f1']:.4f}")
        if dev["macro_f1"] > best["dev_macro_f1"]:
            best = {"dev_macro_f1": dev["macro_f1"], "epoch": epoch,
                    "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}}
    train_seconds = round(time.perf_counter() - start, 1)

    model.load_state_dict(best["state"])
    test_probs = predict(model, test_enc)
    test = scores(test_gold, labels_of(test_probs))
    print(f"seed {seed}: best epoch {best['epoch']} (dev macro F1 {best['dev_macro_f1']:.4f}), "
          f"test accuracy {test['accuracy']:.4f}, test macro F1 {test['macro_f1']:.4f}, {train_seconds} s")
    run = {"seed": seed, "best_epoch": best["epoch"], "dev_macro_f1": best["dev_macro_f1"],
           "history": history, "train_seconds": train_seconds, "test": test}
    return run, model, test_probs

In [ ]:
# The untrained FinBERT on the same test rows, as a check against the README table.
base_model = AutoModelForSequenceClassification.from_pretrained(BASE, revision=BASE_REVISION).to(device)
base_test = scores(test_gold, labels_of(predict(base_model, test_enc)))
print(f"FinBERT before training: test accuracy {base_test['accuracy']:.4f}, macro F1 {base_test['macro_f1']:.4f}")
del base_model

runs, chosen = [], None
for seed in SEEDS:
    run, model, test_probs = train_one(seed)
    runs.append(run)
    if chosen is None or run["dev_macro_f1"] > chosen[0]["dev_macro_f1"]:
        chosen = (run, model.to("cpu"), test_probs)  # picked by dev score only
    else:
        del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

In [ ]:
import csv

chosen_run, chosen_model, chosen_probs = chosen
test_f1 = [r["test"]["macro_f1"] for r in runs]
test_acc = [r["test"]["accuracy"] for r in runs]
summary = {
    "dataset": f"{DATASET}@{DATASET_REVISION[:8]}",
    "base_model": f"{BASE}@{BASE_REVISION[:8]}",
    "smoke_test": SMOKE,
    "sizes": {"train": len(train_texts), "dev": len(dev_texts), "test": len(test_texts)},
    "test_headlines_also_in_training_data": overlap,
    "settings": {"seeds": SEEDS, "epochs": EPOCHS, "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE,
                 "max_tokens": MAX_TOKENS, "warmup_share": WARMUP_SHARE, "weight_decay": WEIGHT_DECAY,
                 "dev_share": DEV_SHARE, "split_seed": SPLIT_SEED, "fp16": USE_FP16},
    "versions": {"python": platform.python_version(), "torch": torch.__version__,
                 "transformers": __import__("transformers").__version__,
                 "datasets": __import__("datasets").__version__, "scikit-learn": __import__("sklearn").__version__},
    "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
    "finbert_before_training": base_test,
    "runs": runs,
    "test_macro_f1_over_seeds": {"mean": round(float(np.mean(test_f1)), 4), "std": round(float(np.std(test_f1, ddof=1)), 4) if len(test_f1) > 1 else None,
                                 "min": min(test_f1), "max": max(test_f1)},
    "test_accuracy_over_seeds": {"mean": round(float(np.mean(test_acc)), 4), "min": min(test_acc), "max": max(test_acc)},
    "saved_model": {"seed": chosen_run["seed"], "best_epoch": chosen_run["best_epoch"],
                    "dev_macro_f1": chosen_run["dev_macro_f1"], "test": chosen_run["test"]},
}
(OUT / "finetune_results.json").write_text(json.dumps(summary, indent=2) + "\n")

save_dir = OUT / "finbert-tweets"
chosen_model.config.id2label = dict(ID2NAME)
chosen_model.config.label2id = dict(LABEL2ID)
chosen_model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

with open(OUT / "test_predictions.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["text", "gold", "predicted"] + [ID2NAME[i] for i in range(len(ID2NAME))])
    for text, gold, row in zip(test_texts, test_gold, chosen_probs):
        writer.writerow([text, gold, ID2NAME[int(row.argmax())]] + [f"{p:.4f}" for p in row])

print(json.dumps({k: summary[k] for k in ("test_macro_f1_over_seeds", "test_accuracy_over_seeds")}, indent=2))
print("saved model: seed", chosen_run["seed"], "epoch", chosen_run["best_epoch"],
      "| test macro F1", chosen_run["test"]["macro_f1"], "| files:", sorted(p.name for p in OUT.iterdir()))